In [ ]:
#THIS CELL CONTAINS THE CODE
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import widgets
from IPython.display import display, clear_output

# --- Core RK2 (Heun's Method) Solver Engine ---
def rk2_step(f, t, y, h):
    k1 = h * f(t, y)
    k2 = h * f(t + h, y + k1)
    return t + h, y + 0.5 * (k1 + k2)

def solve_rk2(f, t_start, t_end, y0, h):
    t_vals, y_vals = [t_start], [y0]
    t, y = t_start, y0
    while t < t_end:
        if t + h > t_end:
            h = t_end - t
        t, y = rk2_step(f, t, y, h)
        t_vals.append(t)
        y_vals.append(y)
    return np.array(t_vals), np.array(y_vals)

# --- Define UI Elements Explicitly ---
f_input = widgets.Text(value='', placeholder='e.g., y * cos(t)', description="dy/dt = f(t,y):")
analytical_input = widgets.Text(value='', placeholder='e.g., exp(sin(t)) (Optional)', description="True y(t):")
y0_input = widgets.FloatText(value=1.0, description="y(t0):")
t_start_input = widgets.FloatText(value=0.0, description="t_start:")
t_end_input = widgets.FloatText(value=10.0, description="t_end:")
h_input = widgets.FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description="Step Size (h):")

# Create manual execute button
btn_update = widgets.Button(description="Update Plot", button_style='success', icon='refresh')
output_area = widgets.Output()

# --- Execution Function ---
def run_simulation(b=None):
    safe_dict = {
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
        "exp": math.exp,
        "log": math.log,
        "sqrt": math.sqrt,
        "pi": math.pi,
        "e": math.e
    }
    
    # Extract values from widgets
    f_str = f_input.value
    analytical_str = analytical_input.value
    y0 = y0_input.value
    t_start = t_start_input.value
    t_end = t_end_input.value
    h = h_input.value
    
    with output_area:
        clear_output(wait=True)  # Instantly clears the old graph safely
        
        if not f_str.strip():
            plt.figure(figsize=(10, 5))
            plt.text(0.5, 0.5, "Please input a function for dy/dt and click 'Update Plot'.", 
                     fontsize=12, ha='center', va='center', color='gray')
            plt.axis('off')
            plt.show()
            return
            
        try:
            f = lambda t, y: eval(f_str, {}, {**safe_dict, "t": t, "y": y})
            t_rk2, y_rk2 = solve_rk2(f, t_start, t_end, y0, h)
            
            plt.figure(figsize=(10, 5))
            plt.plot(t_rk2, y_rk2, 's--', color='crimson', label=f'RK2 Approximation (h={h})', markersize=5)
            
            if analytical_str.strip():
                analytical_f = lambda t: eval(analytical_str, {}, {**safe_dict, "t": t})
                t_fine = np.linspace(t_start, t_end, 500)
                y_true = np.array([analytical_f(t) for t in t_fine])
                plt.plot(t_fine, y_true, color='royalblue', linewidth=2, label='True Analytical Solution')
                
                y_true_at_nodes = np.array([analytical_f(t) for t in t_rk2])
                max_error = np.max(np.abs(y_rk2 - y_true_at_nodes))
                plt.title(f"RK2 vs True Solution | Max Absolute Error: {max_error:.2e}")
            else:
                plt.title("RK2 Approximation (No Analytical Solution Provided)")
                
            plt.xlabel('Time (t)')
            plt.ylabel('State (y)')
            plt.grid(True, linestyle=':', alpha=0.6)
            plt.legend()
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error parsing expressions. Check your operators (e.g., use '*' for multiply).\nDetails: {e}")

# Bind button event to function execution
btn_update.on_click(run_simulation)

# --- Display Dashboard Layout ---
controls = widgets.VBox([f_input, analytical_input, y0_input, t_start_input, t_end_input, h_input, btn_update])
display(controls, output_area)

# Initialize the canvas view on load
run_simulation()


Output()